# Language Processing Laboratory — Session 02
## Regular Expressions and Task-Aware Persian Text Preprocessing

**Duration:** 120 minutes  
**Level:** Undergraduate  
**Prerequisite:** Session 01 (UTF-8, Unicode, basic Persian normalization)  
**Required libraries:** Python standard library only (`re`, `pathlib`, `unicodedata`, `collections`)

> &#x2067;**هدف جلسه:** در جلسهٔ اول فهمیدیم که متن فارسی چگونه در رایانه نمایش داده می‌شود. امروز با **عبارت‌های منظم (Regular Expressions)** الگوهایی را از متن استخراج می‌کنیم و یک خط لولهٔ پیش‌پردازش می‌سازیم که قابل‌آزمایش و متناسب با هدف مسئله باشد.&#x2069;

**This is a student starter notebook.** Complete the `TODO` code and **✍️ Exercise** explanations yourself. No instructor solutions are included.

## Learning outcomes, repository, and submission

Read `docs/AI_USE_POLICY_FA.md` and `docs/SUBMISSION_FA.md` from the repository root. Do not publish completed student notebooks or answer keys in a public GitHub repository.

By the end of this session you should be able to:

1. distinguish literal, character-class, quantifier and anchor patterns in Python `re`;
2. use `re.search`, `re.findall`, `re.finditer` and `re.sub` appropriately;
3. extract simple URLs, email addresses, hashtags and digit sequences from **Persian** text;
4. identify over-matching and under-matching of regex patterns;
5. normalize whitespace **without discarding ZWNJ or line boundaries**;
6. implement and test a conservative, **idempotent** Persian preprocessing policy;
7. quantify changes and justify which linguistic signals should be preserved.

**Not covered today:** production-ready URL/email validation, full word tokenization, stemming and lemmatization. Dedicated tokenization work belongs to a different lab.

## Lab workflow and assessment

| Time | Activity |
|---:|---|
| 0–10 min | Reproducibility, review of Session 01, load corpus |
| 10–30 min | Regex building blocks and Unicode digits |
| 30–55 min | Regex-based information extraction |
| 55–75 min | Line-aware whitespace cleanup + ZWNJ |
| 75–100 min | Task-aware preprocessing pipeline + public checks |
| 100–112 min | Before/after evidence and error analysis |
| 112–120 min | Exit ticket, submission check |

**Rubric (10 points):** 1 pt reproducibility/metadata; 2 pts regex understanding + extraction; 2 pts whitespace handling; 2 pts preprocessing policy and public checks; 2 pts quantitative evaluation and error analysis; 1 pt exit ticket/clean submission.

Save your work as `STUDENT_ID_session02.ipynb`. Run the notebook **from a fresh kernel** before submitting. The public checks are formative; a failing check in this *starter* notebook is expected until you implement the TODOs.

## 0. Environment and student metadata

Run this cell unchanged. The corpus uses only fictional or generic examples. It contains no collected personal information.

In [ ]:
import sys
import re
import unicodedata
from pathlib import Path
from collections import Counter

print('Python:', sys.version.split()[0])
print('Regex engine: Python standard-library re')
if sys.version_info < (3, 10):
    print('WARNING: Python 3.10+ recommended.')

In [ ]:
# TODO: Exercise 0 — replace the empty strings.
student_id = ''
student_name = ''
university = ''

print('Student ID:', student_id)
print('Name:', student_name)
print('University:', university)

### Load the Session 02 mini-corpus

The data file is intentionally **raw and inconsistent**. Do not edit it. Always read the file using UTF-8.

**Notebook working directory:** open this notebook from `sessions/02_regex_preprocessing/` in JupyterLab. If the path below fails, verify the folder structure before changing any code.

In [ ]:
DATA_PATH = Path('data/raw_persian_messages.txt')
raw_messages = DATA_PATH.read_text(encoding='utf-8').splitlines()
print(f'{len(raw_messages)} messages loaded from {DATA_PATH}')
for index, message in enumerate(raw_messages[:5], start=1):
    print(f'{index:02d}: {message}')

---
# 1. Regex concepts: patterns are not “magic cleaning”

Python's [`re` module](https://docs.python.org/3/library/re.html) supports patterns such as:

| Regex | Meaning | Example |
|---|---|---|
| `\d` | any Unicode decimal digit (unless ASCII flag) | `۲` and `2` |
| `[0-9]` | ASCII digits only | `2` but not `۲` |
| `[۰-۹]` | Persian digits | `۲` |
| `+` | one or more | `سلام+` (one or more final `م`) |
| `\s` | whitespace, **including newline** | ` ` and `\n` |
| `^` / `$` | start / end of string or line under flags | anchored pattern |

Always use raw strings such as `r'\d+'` for patterns. **Important:** `\w` and `\b` are Unicode-aware but do not necessarily correspond to Persian linguistic word boundaries; ZWNJ (`U+200C`) is not matched by `\w`.

In [ ]:
text = 'تعداد ۲۵ کتاب و 17 مقاله در سال ٢٠٢٦ گزارش شد.'
print('Unicode decimal digits:', re.findall(r'\d+', text))
print('ASCII digits only:   ', re.findall(r'[0-9]+', text))
print('Persian digits only: ', re.findall(r'[۰-۹]+', text))
print('Arabic-Indic only:   ', re.findall(r'[٠-٩]+', text))

# search finds the first match; finditer exposes spans as well.
match = re.search(r'\d+', text)
print('First digit match:', match.group(0), 'at', match.span())
print('Every match and span:')
for item in re.finditer(r'\d+', text):
    print(item.group(0), item.span())

### ✍️ Exercise 1 — Regex predictions and explanations

Before running any new code, predict what each pattern will match in:

`'دانشجو_۱ #پردازش_زبان می‌رود 123'`

1. `r'\d+'`
2. `r'[0-9]+'`
3. `r'#\w+'`

Then explain in **2–4 sentences** why `#\w+` may miss part of a hashtag containing a ZWNJ. Show one counterexample using `re.findall`.

> TODO: write your prediction, observed output, and explanation here.

In [ ]:
# TODO: Exercise 1 — experiment and verify your predictions.
experiment = 'دانشجو_۱ #پردازش_زبان می‌رود 123'
patterns = [r'\d+', r'[0-9]+', r'#\w+']
for pattern in patterns:
    print(pattern, '->', re.findall(pattern, experiment))

# Add a ZWNJ-containing hashtag and inspect it.

---
# 2. Information extraction (استخراج الگو)

We are **extracting features**, not validating identities or performing full tokenization.

Demonstration: basic email / URL / hashtag regex patterns. These are **intentionally simple**. They are useful for this controlled corpus, but not comprehensive validators for the open web.

In [ ]:
example = 'راهـنما: https://example.org/faq و contact@example.org #پردازش_زبان #می‌روم'

url_pattern = r'https?://[^\s<>]+'
email_pattern = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
hashtag_pattern = r'#[\w\u200c]+'

print('URLs:', re.findall(url_pattern, example))
print('Emails:', re.findall(email_pattern, example))
print('Hashtags:', re.findall(hashtag_pattern, example))
print('Digit groups:', re.findall(r'\d+', 'سال ۱۴۰۵ و سال 2026'))

### ✍️ Exercise 2 — Implement `extract_patterns`

Write `extract_patterns(text)` to return **exactly four keys**:

- `urls`: `http://` or `https://` links in this course corpus;
- `emails`: simple ASCII-form email addresses (no internationalized email validation);
- `hashtags`: a leading `#` followed by word characters or ZWNJ;
- `digit_groups`: all consecutive Unicode decimal-digit runs (`\d+`).

Return lists of **matched strings** in left-to-right order. Your URL pattern should **not keep sentence-final `.` or `،` or `!` or `؟`**. These rules describe a *teaching extractor*, not a production-grade parser.

Use `re.findall` and/or `re.finditer`. Do not hard-code expected outputs.

In [ ]:
def extract_patterns(text):
    """Return lists under urls/emails/hashtags/digit_groups (see spec)."""
    # TODO: Exercise 2 — replace the placeholder values.
    return {
        'urls': [],
        'emails': [],
        'hashtags': [],
        'digit_groups': [],
    }

print(extract_patterns('پست https://example.org/news، ایمیل hello@example.org #زبان ۱۴۰۵'))

### Public checks — pattern extraction

Do **not** delete or edit these checks. They print `PASS`/`FAIL` instead of raising errors, so the original starter notebook can be executed safely. Complete your TODOs until all checks pass.

In [ ]:
def check_extraction(extract_fn):
    cases = [
        ('https://example.org/a. به hi@example.org #زبان ۱۴۰۵',
         {'urls':['https://example.org/a'], 'emails':['hi@example.org'],
          'hashtags':['#زبان'], 'digit_groups':['۱۴۰۵']}),
        ('#می‌روم #پردازش_زبان سال 2026',
         {'urls':[], 'emails':[], 'hashtags':['#می‌روم', '#پردازش_زبان'],
          'digit_groups':['2026']}),
        ('بدون پیوند و ایمیل',
         {'urls':[], 'emails':[], 'hashtags':[], 'digit_groups':[]}),
    ]
    passed = 0
    for index, (text, expected) in enumerate(cases, 1):
        try:
            observed = extract_fn(text)
            ok = observed == expected
        except Exception as exc:
            observed = f'ERROR: {exc}'
            ok = False
        print(('PASS' if ok else 'FAIL'), f'case {index}', repr(observed))
        passed += int(ok)
    print(f'Extraction: {passed}/{len(cases)} public checks passed')
    return passed == len(cases)

check_extraction(extract_patterns)

### ✍️ Exercise 2b — Model limitations

Describe **one false positive** and **one false negative** that could occur when using these simple patterns on real Persian social-media data. Explain why regex extraction is not the same as linguistic tokenization.

> TODO: 3–5 sentences here.

---
# 3. Whitespace and ZWNJ: clean *selectively*

`re.sub(r'\s+', ' ', text)` is often **too destructive** because `\s` also matches newlines. When each line is a separate document, collapsing all whitespace merges document boundaries.

In this section, treat each line as one document and preserve `\n` between documents. Never replace or remove ZWNJ (`U+200C`) silently.

In [ ]:
spacing_example = '  سلام   دنیا\t!\n  می‌روم  به   دانشگاه  '
naive = re.sub(r'\s+', ' ', spacing_example).strip()
print('Original:', repr(spacing_example))
print('Naive:', repr(naive))
print('Newline preserved?', '\n' in naive)
print('ZWNJ preserved?', '\u200c' in naive)

### ✍️ Exercise 3 — Implement `normalize_spacing`

Requirements:

1. For **each line separately**, collapse runs of spaces or tabs to one ordinary space.
2. Strip spaces/tabs at each line boundary.
3. Preserve newline separators and **empty lines** (so line count does not change).
4. Preserve ZWNJ, punctuation, digits, and emoji.
5. The function should be idempotent: applying it twice has the same result.

Hint: use `text.split('\n')`, `re.sub(r'[ \t]+', ' ', line)` and `'\n'.join(...)`.

In [ ]:
def normalize_spacing(text):
    """Normalize ordinary spaces/tabs independently per line (see spec)."""
    # TODO: Exercise 3
    return text

In [ ]:
def check_spacing(spacing_fn):
    cases = [
        ('  داده   خوب\tاست  ', 'داده خوب است'),
        ('  خط  اول  \n  خط\tدوم ', 'خط اول\nخط دوم'),
        ('سلام\n\n  می‌روم   !', 'سلام\n\nمی‌روم !'),
    ]
    passed = 0
    for index, (text, expected) in enumerate(cases, 1):
        try:
            got = spacing_fn(text)
            ok = got == expected and spacing_fn(got) == got
        except Exception as exc:
            got, ok = f'ERROR: {exc}', False
        print(('PASS' if ok else 'FAIL'), f'case {index}', repr(got))
        passed += int(ok)
    print(f'Spacing: {passed}/{len(cases)} public checks passed')
    return passed == len(cases)

check_spacing(normalize_spacing)

---
# 4. A conservative task-aware Persian preprocessing pipeline

Recall Session 01: NFC Unicode normalization does **not** automatically map Arabic `ي/ى` or `ك` to Persian `ی`/`ک`.

For a **Persian news/topic-classification corpus**, use this explicit policy:

1. apply NFC;
2. map `ي` and `ى` to `ی`, and `ك` to `ک`;
3. normalize ordinary spaces/tabs **per line**, retaining line breaks;
4. **preserve** ZWNJ, punctuation, hashtags, URLs, emails, digits, and emoji;
5. do not remove URLs, emojis or stopwords without an experiment showing a benefit.

This policy is **not universal**. For a privacy-sensitive task, addresses, emails, or URLs may require redaction—not just normalization. We keep them in this synthetic teaching data for analysis.

### ✍️ Exercise 4 — Implement `preprocess_persian`

Combine `unicodedata.normalize('NFC', ...)`, the specified character replacements, and your `normalize_spacing` function. Complete all TODOs without copying code from solved notebooks.

In [ ]:
def preprocess_persian(text):
    """Conservative Persian preprocessing; implement the policy above."""
    # TODO: Exercise 4
    return text

In [ ]:
def check_preprocessing(preprocess_fn):
    cases = [
        ('  يك   كتاب درباره يادگيري  ماشين  ', 'یک کتاب درباره یادگیری ماشین'),
        ('مي‌روم\n  كد  2026 #زبان', 'می‌روم\nکد 2026 #زبان'),
        ('ا\u0653زمایش  خوب است!', 'آزمایش خوب است!'),
        ('  سلام\n\n  متن  ', 'سلام\n\nمتن'),
        ('www.example.org / https://example.org/a #دانشگاه 😀',
         'www.example.org / https://example.org/a #دانشگاه 😀'),
    ]
    passed = 0
    for index, (text, expected) in enumerate(cases, 1):
        try:
            got = preprocess_fn(text)
            ok = got == expected and preprocess_fn(got) == got
        except Exception as exc:
            got, ok = f'ERROR: {exc}', False
        print(('PASS' if ok else 'FAIL'), f'case {index}', repr(got))
        passed += int(ok)
    print(f'Preprocessing: {passed}/{len(cases)} public checks passed')
    return passed == len(cases)

check_preprocessing(preprocess_persian)

### ✍️ Exercise 4b — Justify the policy

In 3–5 sentences:

- Why should ZWNJ be preserved in this dataset?
- Why should `\n` be preserved instead of replaced by a space?
- Give one realistic NLP task where removing punctuation would be harmful.

> TODO: write your reasoning here.

---
# 5. Measurement and error analysis

**Never claim a preprocessing step is beneficial just because its output looks cleaner.** Collect evidence first. The course corpus is small and synthetic, so any result is a **demonstration**, not a general performance guarantee.

First inspect some real input lines and their outputs:

In [ ]:
for message in raw_messages[:5]:
    print('RAW: ', repr(message))
    print('PREP:', repr(preprocess_persian(message)))
    print('-' * 60)

### ✍️ Exercise 5 — Quantitative comparison

For the original messages and your normalized messages, compute and report:

1. number of occurrences of Arabic `ي`, `ى`, and `ك`;
2. number of ZWNJs (`U+200C`);
3. number of tabs and repeated ordinary-space runs (`r' {2,}'`);
4. number of extracted URLs, emails and hashtags **before and after** your preprocessing.

State which counts **should** be invariant under this policy, and which **should** change. Use the functions you implemented; do not hard-code counts.

**Important:** In regex outputs, a character count and a match count are different quantities. Label your columns accurately.

In [ ]:
def measure_corpus(messages):
    """Return quantitative diagnostics for a sequence of strings."""
    # TODO: Exercise 5
    return {}

raw_metrics = measure_corpus(raw_messages)
cleaned_messages = [preprocess_persian(m) for m in raw_messages]
clean_metrics = measure_corpus(cleaned_messages)
print('Raw:    ', raw_metrics)
print('Cleaned:', clean_metrics)

### ✍️ Exercise 5b — Evidence-based interpretation

Write a brief research-style comparison (**5–7 sentences**):

- Identify **two measured changes** and **one expected invariant**.
- Show **one extracted pattern** for which your regex would make a mistake outside this corpus (for example, a URL followed by unusual punctuation).
- Propose one improvement. What extra test case would verify it?
- Explicitly state whether these results prove higher accuracy on a downstream task (and why or why not).

> TODO: your quantitative interpretation and error analysis here.

---
# 6. Exit ticket

Answer concisely in your own words:

1. What is the difference between `re.search` and `re.findall`?
2. Why does `\s+` need care when each line is a separate document?
3. Why does `\d` recognize more characters than `[0-9]` in Python 3?
4. What does *idempotent preprocessing* mean?
5. When might it be necessary to **redact** an email rather than preserve it?

> TODO: write your answers here.

# Final submission checklist

- [ ] Exercise 0 metadata completed.
- [ ] Exercises 1–5 and written explanations completed.
- [ ] `extract_patterns`, `normalize_spacing`, and `preprocess_persian` pass every **public check**.
- [ ] `measure_corpus` returns computed results for both versions of the dataset.
- [ ] Explain one regex limitation and one context-dependent preprocessing trade-off.
- [ ] Notebook runs via **Kernel → Restart Kernel and Run All Cells**.
- [ ] Filename: `STUDENT_ID_session02.ipynb`.
- [ ] Do **not** push a completed solution to a public repository.

**External help / sources used:**  
> TODO: cite any non-trivial external code, idea, or AI assistance here, per `docs/AI_USE_POLICY_FA.md`.

**End of Session 02.**